In [5]:
# Question1:
# If you have trained five different models on the exact same training data, and
# they all achieve 95% precision, is there any chance that you can combine these
# models to get better results? If so, how? If not, why?

#Yes, it is possible to combine models that achieve similar precision in order to improve overall performance (e.g., accuracy, recall, or precision). One common approach is to use ensemble methods, such as a voting classifier, where predictions from multiple models are aggregated and the final prediction is based on the majority vote.

#A key assumption behind this approach is that the models are diverse, meaning they capture different patterns in the data and make different errors. When the models’ errors are not highly correlated, combining them can reduce overall error and improve performance.

#However, if the classifiers are very similar—sharing the same assumptions and making highly correlated errors—then the ensemble is unlikely to provide any benefit.

#While majority voting is a simple and common method, there are several other ways to combine models, including:

#Averaging predicted probabilities
#Stacking (using a meta-model to combine predictions)
#Bagging and boosting methods

In [8]:
# Question 2. 
# What is the difference between hard and soft voting classifiers?
# hard voting classifier aggregate the predictions of each classifier and predict the class that gets the most votes.
# while soft voting classifier works on classifiers that can estimate class probabilities and uses these classe probabilities to If all classifiers are able to estimate class probabilities to predict the class with the
# highest class probability, averaged over all the individual classifiers.

In [11]:
# Question 3:
#  Is it possible to speed up training of a bagging ensemble by distributing it
# across multiple servers? What about pasting ensembles, boosting ensembles,
# Random Forests, or stacking ensembles?

# Yes the biggested advantage of bagging ensemble is that it trains different classifiers with replacement, the same is done with pasting but
# the idea here is that subsamples of the training data are given to different classifiers withOUT replacement, both approaches can be run in parallel
# Boosting ensembles (e.g., AdaBoost, Gradient Boosting) takes the predictors of the previous training step in order to fit new predictors to the individual errors, so there is a sequential dependence inhereted by the approach itself that prevents parallelization
# Random Forests can also be parallelized. Each tree in the forest is trained independently (like bagging, but with additional feature randomness), so training can be distributed across servers.
# Stacking ensembles are partially parallelizable. The base-level models can be trained in parallel, but the meta-model depends on their predictions, so there is a sequential step involved.


In [ ]:
# Question 4. What is the benefit of out-of-bag evaluation?
#Out-of-bagging evaluation uses training that has never been used during the training model to evaluate the performance of the ensemble
# the advantage of using this approach is reliable estimate of model performance without 
# requiring a separate validation set, allowing you to make full use of the available training data. 
# This is especially useful when data is limited.


#5. What makes Extra-Trees more random than regular Random Forests? How can
#this extra randomness help? Are Extra-Trees slower or faster than regular
#Random Forests?

# Extra-trees are done so when making trees by using random thresholds for each feature rather than
# searching for the best possible thresholds. This technique trades more bias for lower variance which would 
# tend to improves model's generalization performance—especially by helping prevent overfitting.
# Extra-trees are much faster than random forest because one does not need to find the 
# best possible threshold for every node of the tree, which is a timing consuming task.


#6. If your AdaBoost ensemble underfits the training data, which hyperparameters
# should you tweak and how?
# One way for a new predictor to correct its predecessor is to pay a bit more attention to
#the training instances that the predecessor underfitted.
# The algorithm then increases the relative weight of misclassified training instances.
# Then it trains a second classifier, using the updated weights, and again makes
# predictions on the training set, updates the instance weights.
# The parameters that one shall tune are the learning rate, 
# If AdaBoost underfits, increase the number of estimators, use more complex base learners,
# or increase the learning rate to allow the model to better fit the training data.
    

# 7. If your Gradient Boosting ensemble overfits the training set, should you
# increase or decrease the learning rate?
# Decrease the learning rate, this makes each tree’s contribution smaller, reducing overfitting and improving generalization.




9. Run the individual classifiers from the previous exercise to make predictions
on the validation set, and create a new training set with the resulting
predictions: each training instance is a vector containing the set of predictions
from all your classifiers for an image, and the target is the image’s class. Train
a classifier on this new training set. Congratulations, you have just trained a
blender, and together with the classifiers it forms a stacking ensemble! Now
evaluate the ensemble on the test set. For each image in the test set, make
predictions with all your classifiers, then feed the predictions to the blender to
get the ensemble’s predictions. How does it compare to the voting classifier
you trained earlier?

In [ ]:
# 8. Load the MNIST data (introduced in Chapter 3), and split it into a training set,
# a validation set, and a test set (e.g., use 50,000 instances for training, 10,000
# for validation, and 10,000 for testing). Then train various classifiers, such as a
# Random Forest classifier, an Extra-Trees classifier, and an SVM classifier.
# Next, try to combine them into an ensemble that outperforms each individual
# classifier on the validation set, using soft or hard voting. Once you have found
# one, try it on the test set. How much better does it perform compared to the
# individual classifiers?
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Getting the data
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist["data"], mnist["target"]


# Splitting the data
# First split: train (50k) + temp (20k)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=20000, random_state=42)
# Validation data
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

log_clf = LogisticRegression()
rnd_clf = RandomForestClassifier()
svm_clf = SVC()

voting_hard_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)], voting='hard'
)

voting_hard_clf.fit(X_train, y_train)

from sklearn.metrics import accuracy_score
for clf in (log_clf, rnd_clf, svm_clf, voting_hard_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))
